# Building Footprint Segmentation from Aerial Imagery
**PhD Technical Assessment -- Part 2**

This notebook builds an end-to-end binary semantic segmentation pipeline to extract building
footprints from high-resolution aerial imagery, using the Inria Aerial Image Labeling Dataset.

**Note on dataset structure:** the dataset provides 180 labeled training tiles across 5 cities
(Austin, Chicago, Kitsap County, Western Tyrol, Vienna; 36 tiles each) and a further 180 test
tiles whose ground-truth masks were never publicly released (the dataset originated as a
competition). All training and evaluation in this notebook therefore uses only the 180 labeled
training tiles, split into our own train/validation partition (see Task partitioning below).

## Setup

In [ ]:
import os
import re
import time
import numpy as np
import matplotlib.pyplot as plt
import tifffile
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

device = torch.device('xpu' if torch.xpu.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu'))
print("Using device:", device)

## Data Exploration

Before building the pipeline, we inspect one tile and its mask to confirm the data format
matches the assessment's specification (5000x5000 RGB tiles; masks with 255=building,
0=background).

In [ ]:
image_dir = 'AerialImageDataset/train/images'
mask_dir = 'AerialImageDataset/train/gt'

sample_image = tifffile.imread(os.path.join(image_dir, 'austin1.tif'))
sample_mask = tifffile.imread(os.path.join(mask_dir, 'austin1.tif'))

print("Image shape:", sample_image.shape, "dtype:", sample_image.dtype)
print("Mask shape:", sample_mask.shape, "dtype:", sample_mask.dtype)
print("Unique mask values:", np.unique(sample_mask))

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(sample_image); axes[0].set_title('Aerial image (austin1.tif)'); axes[0].axis('off')
axes[1].imshow(sample_mask, cmap='gray'); axes[1].set_title('Ground truth mask'); axes[1].axis('off')
plt.tight_layout()
plt.savefig('fig_sample_tile.png', dpi=150)
plt.show()

## Data Partitioning: City-Level Train/Validation Split

Rather than randomly shuffling all 180 tiles into train/validation, we hold out **one entire
city** for validation. This directly prevents geographic leakage: random shuffling could let
near-identical neighboring patches from the same city end up split across train and validation,
inflating apparent performance by testing on visually familiar geography rather than genuinely
unseen locations. Holding out a whole city instead tests the model's ability to generalize to
architecture, layout, and imagery conditions it has never encountered even once during
training -- a stricter and more honest evaluation, and one that mirrors the original dataset's
own design (which similarly withheld entire cities for its official test set).

In [ ]:
all_files = sorted(os.listdir(image_dir))

def get_city(filename):
    return re.match(r'([a-zA-Z]+)', filename).group(1)

cities = sorted(set(get_city(f) for f in all_files))
print("Cities found:", cities)
for city in cities:
    count = sum(1 for f in all_files if get_city(f) == city)
    print(f"  {city}: {count} tiles")

VAL_CITY = 'chicago'
train_files = [f for f in all_files if get_city(f) != VAL_CITY]
val_files = [f for f in all_files if get_city(f) == VAL_CITY]

print(f"\nTrain tiles: {len(train_files)} (cities: {sorted(set(get_city(f) for f in train_files))})")
print(f"Val tiles:   {len(val_files)} (held-out city: {VAL_CITY})")

## Feature/Patch Pipeline

Each 5000x5000 tile is too large to feed directly into a segmentation network, so we extract
smaller 512x512 patches. Patch extraction happens strictly **after** the city-level split above,
so no patch derived from a training-city tile can ever be adjacent to, or overlap with, a patch
from the held-out validation city.

Rather than pre-extracting and storing every possible patch (memory- and disk-intensive for
tiles this large), we use a generator that reads each tile from disk **exactly once** and
extracts multiple random patches from it before moving to the next tile -- this avoids the
severe I/O bottleneck of re-opening large TIFFs repeatedly, while remaining memory-safe
regardless of available RAM.

In [ ]:
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

In [ ]:
def tile_batch_generator(filenames, image_dir, mask_dir, patch_size, patches_per_tile,
                          batch_size, transform, shuffle_tiles=True):
    """Yields batches of augmented (image, mask) patch tensors, reading each tile from disk
    exactly once, then extracting multiple random patches from it before moving to the next
    tile -- avoids re-reading the same large tile repeatedly."""
    file_order = list(filenames)
    if shuffle_tiles:
        np.random.shuffle(file_order)

    patch_buffer_imgs, patch_buffer_masks = [], []

    for filename in file_order:
        image = tifffile.imread(os.path.join(image_dir, filename))
        mask = tifffile.imread(os.path.join(mask_dir, filename))
        h, w = mask.shape

        for _ in range(patches_per_tile):
            top = np.random.randint(0, h - patch_size)
            left = np.random.randint(0, w - patch_size)
            img_patch = image[top:top+patch_size, left:left+patch_size]
            mask_patch = mask[top:top+patch_size, left:left+patch_size]
            mask_patch = (mask_patch > 0).astype(np.float32)

            augmented = transform(image=img_patch, mask=mask_patch)
            patch_buffer_imgs.append(augmented['image'])
            patch_buffer_masks.append(augmented['mask'])

            if len(patch_buffer_imgs) == batch_size:
                yield torch.stack(patch_buffer_imgs), torch.stack(patch_buffer_masks)
                patch_buffer_imgs, patch_buffer_masks = [], []

    if patch_buffer_imgs:
        yield torch.stack(patch_buffer_imgs), torch.stack(patch_buffer_masks)

PATCH_SIZE = 512
PATCHES_PER_TILE = 16
BATCH_SIZE = 8

steps_per_train_epoch = (len(train_files) * PATCHES_PER_TILE) // BATCH_SIZE
steps_per_val_epoch = (len(val_files) * PATCHES_PER_TILE) // BATCH_SIZE
print("Train steps/epoch:", steps_per_train_epoch, " Val steps/epoch:", steps_per_val_epoch)

## Model Architecture

We use a **U-Net** with a ResNet34 encoder pretrained on ImageNet. U-Net's encoder-decoder
structure with skip connections is a standard, well-justified choice for segmentation: the
encoder progressively downsamples to capture *what* is present, the decoder progressively
upsamples back to full resolution to recover *where* it is, and the skip connections carry fine
spatial detail (critical for precise building edges) directly from encoder to decoder layers,
bypassing the information loss that repeated downsampling would otherwise cause. The
ImageNet-pretrained encoder provides transfer learning, giving faster convergence and typically
better results than training from random initialization, particularly valuable given a limited
training budget.

In [ ]:
model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1,
    activation=None,  # raw logits; sigmoid applied inside the loss for numerical stability
)
model = model.to(device)

print("Model created. Total parameters:", sum(p.numel() for p in model.parameters()))

## Loss Function: Combined Dice + BCE

Buildings typically cover a minority of pixels in a tile, creating class imbalance. Plain
binary cross-entropy would let a model achieve deceptively low loss by favoring the background
majority. Dice loss instead measures overlap between predicted and actual building regions,
normalized by their combined size, making it robust to this imbalance -- a model predicting
"background everywhere" scores a Dice loss near its maximum penalty rather than a
deceptively good score. Dice loss's gradients can be unstable early in training when predictions
are near all-zero, so it is combined with BCE, which provides smoother per-pixel gradients
throughout training. This is a standard, well-established combination in segmentation
literature.

In [ ]:
class DiceBCELoss(nn.Module):
    def __init__(self, bce_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.bce_weight = bce_weight

    def dice_loss(self, logits, targets, smooth=1.0):
        probs = torch.sigmoid(logits)
        probs = probs.view(-1)
        targets = targets.view(-1)
        intersection = (probs * targets).sum()
        dice_coef = (2. * intersection + smooth) / (probs.sum() + targets.sum() + smooth)
        return 1 - dice_coef

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        d_loss = self.dice_loss(logits, targets)
        return self.bce_weight * bce_loss + (1 - self.bce_weight) * d_loss

criterion = DiceBCELoss(bce_weight=0.5)

## Evaluation Metrics: Beyond Pixel Accuracy

Pixel accuracy is misleading for imbalanced segmentation: if only ~15% of pixels are building,
a trivial "predict background everywhere" model already scores ~85% pixel accuracy while being
useless. We report it only for contrast, alongside two metrics standard in segmentation
literature: **IoU** (Intersection over Union -- the ratio of correctly predicted building area
to the union of predicted and actual building area) and **F1/Dice score** (mathematically
equivalent to the Dice coefficient for binary segmentation, weighting precision and recall
equally). A trivial background-only model scores 0 on both, correctly exposing it as
uninformative -- unlike pixel accuracy.

In [ ]:
def compute_metrics(logits, targets, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()

    preds_flat = preds.view(-1)
    targets_flat = targets.view(-1)

    intersection = (preds_flat * targets_flat).sum()
    union = preds_flat.sum() + targets_flat.sum() - intersection

    iou = (intersection + 1e-7) / (union + 1e-7)
    dice_f1 = (2 * intersection + 1e-7) / (preds_flat.sum() + targets_flat.sum() + 1e-7)
    pixel_acc = (preds_flat == targets_flat).float().mean()

    return iou.item(), dice_f1.item(), pixel_acc.item()

## Training

10 epochs, Adam optimizer, learning rate 1e-4. Training and validation loss/metrics are tracked
each epoch for later inspection. On an Intel Arc A580 GPU (via PyTorch's XPU backend), one epoch
takes approximately 10-12 minutes with this data pipeline and patch configuration.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
NUM_EPOCHS = 10
PRINT_EVERY = 20

history = {'train_loss': [], 'val_loss': [], 'val_iou': [], 'val_f1': []}

for epoch in range(NUM_EPOCHS):
    start_time = time.time()

    model.train()
    train_loss_sum, train_batches = 0.0, 0
    train_gen = tile_batch_generator(train_files, image_dir, mask_dir, PATCH_SIZE,
                                       PATCHES_PER_TILE, BATCH_SIZE, train_transform, shuffle_tiles=True)
    for images, masks in train_gen:
        images = images.to(device)
        masks = masks.unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item()
        train_batches += 1
        if train_batches % PRINT_EVERY == 0:
            print(f"  [Epoch {epoch+1}] train batch {train_batches}/{steps_per_train_epoch}, "
                  f"running avg loss: {train_loss_sum/train_batches:.4f}, "
                  f"elapsed: {time.time()-start_time:.0f}s")

    train_loss = train_loss_sum / train_batches

    model.eval()
    val_loss_sum, val_iou_sum, val_f1_sum, val_batches = 0.0, 0.0, 0.0, 0
    val_gen = tile_batch_generator(val_files, image_dir, mask_dir, PATCH_SIZE,
                                     PATCHES_PER_TILE, BATCH_SIZE, val_transform, shuffle_tiles=False)
    with torch.no_grad():
        for images, masks in val_gen:
            images = images.to(device)
            masks = masks.unsqueeze(1).to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)
            iou, f1, _ = compute_metrics(outputs, masks)
            val_loss_sum += loss.item()
            val_iou_sum += iou
            val_f1_sum += f1
            val_batches += 1

    val_loss = val_loss_sum / val_batches
    val_iou = val_iou_sum / val_batches
    val_f1 = val_f1_sum / val_batches

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_iou'].append(val_iou)
    history['val_f1'].append(val_f1)

    elapsed = time.time() - start_time
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val IoU: {val_iou:.4f} | Val F1: {val_f1:.4f} | "
          f"Time: {elapsed:.1f}s\n")

## Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(range(1, NUM_EPOCHS+1), history['train_loss'], marker='o', label='Train Loss')
axes[0].plot(range(1, NUM_EPOCHS+1), history['val_loss'], marker='o', label='Val Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()

axes[1].plot(range(1, NUM_EPOCHS+1), history['val_iou'], marker='o', color='green', label='Val IoU')
axes[1].plot(range(1, NUM_EPOCHS+1), history['val_f1'], marker='o', color='darkorange', label='Val F1')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Score')
axes[1].set_title('Validation IoU and F1 (Chicago, held-out city)')
axes[1].legend()

plt.tight_layout()
plt.savefig('fig_training_curves.png', dpi=150)
plt.show()

**Discussion:** training loss falls smoothly and monotonically across all 10 epochs, confirming
consistent learning. Validation metrics, by contrast, are noticeably noisier -- notably a sharp
dip at epoch 6 (IoU falling from 0.60 back to 0.49) before recovering to the best result at
epoch 10 (IoU 0.64, F1 0.78, lowest validation loss of the run). This pattern suggests the
learning rate may be slightly high for fully smooth convergence, and that performance was still
improving at the final epoch rather than having clearly plateaued -- additional epochs may have
yielded further gains, which is noted here as a limitation of the fixed 10-epoch budget rather
than evidence that epoch 10 represents a true optimum.

## Qualitative Evaluation: Success and Failure Cases

We generate predictions on fixed, centered patches from six held-out Chicago tiles (fixed rather
than random locations, for reproducible, discussable examples), then visualize the original
image, ground truth mask, and predicted mask side by side for a strong success case and a clear
failure case.

In [ ]:
model.eval()

def get_prediction_examples(filenames, image_dir, mask_dir, model, patch_size, n_examples=6):
    examples = []
    with torch.no_grad():
        for filename in filenames[:n_examples]:
            image = tifffile.imread(os.path.join(image_dir, filename))
            mask = tifffile.imread(os.path.join(mask_dir, filename))
            h, w = mask.shape

            top, left = (h - patch_size) // 2, (w - patch_size) // 2
            img_patch = image[top:top+patch_size, left:left+patch_size]
            mask_patch = (mask[top:top+patch_size, left:left+patch_size] > 0).astype(np.float32)

            augmented = val_transform(image=img_patch, mask=mask_patch)
            img_tensor = augmented['image'].unsqueeze(0).to(device)

            output = model(img_tensor)
            pred_prob = torch.sigmoid(output).cpu().numpy()[0, 0]
            pred_mask = (pred_prob > 0.5).astype(np.float32)

            iou, f1, _ = compute_metrics(output, augmented['mask'].unsqueeze(0).unsqueeze(0).to(device))
            examples.append((img_patch, mask_patch, pred_mask, iou, f1, filename))
    return examples

examples = get_prediction_examples(val_files, image_dir, mask_dir, model, PATCH_SIZE, n_examples=6)

for img, gt, pred, iou, f1, fname in examples:
    print(f"{fname}: IoU={iou:.3f}, F1={f1:.3f}")

In [ ]:
# Select the strongest and weakest examples by IoU for the success/failure comparison
examples_sorted = sorted(examples, key=lambda e: e[3], reverse=True)
success_examples = examples_sorted[:1]
failure_examples = examples_sorted[-1:]
display_examples = success_examples + failure_examples

fig, axes = plt.subplots(len(display_examples), 3, figsize=(11, 4 * len(display_examples)))

for row, (img, gt, pred, iou, f1, fname) in enumerate(display_examples):
    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f'{fname}\nOriginal image')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(gt, cmap='gray')
    axes[row, 1].set_title('Ground truth mask')
    axes[row, 1].axis('off')

    axes[row, 2].imshow(pred, cmap='gray')
    axes[row, 2].set_title(f'Predicted mask\nIoU={iou:.3f}, F1={f1:.3f}')
    axes[row, 2].axis('off')

plt.tight_layout()
plt.savefig('fig_qualitative_examples.png', dpi=150)
plt.show()

**Discussion:** the success case (a large, simple, high-contrast industrial building) is
segmented almost pixel-perfectly, including a small isolated structure. The failure case (a
dense residential block of small, closely-packed row houses) reveals a specific, explainable
failure mode: the model correctly identifies roughly where buildings are, but merges
closely-spaced adjacent small buildings into single connected blobs, losing the narrow gaps
between individual structures that the ground truth preserves. This is consistent with a known
limitation of encoder-decoder segmentation architectures: the ResNet34 encoder's progressive
downsampling loses fine spatial resolution before the decoder attempts to recover it, and narrow
gaps between tightly-packed small structures are exactly the kind of fine detail most vulnerable
to this loss. Skip connections partially mitigate this for large, well-separated structures (as
seen in the success case) but are less effective when structures are only a few pixels apart to
begin with.

## Loss Function Ablation: Explicit Boundary Weighting

Dice loss provides some indirect sensitivity to boundary overlap, but as a deliberate test of
whether a more explicit mechanism improves edge precision further, a boundary-weighted variant
was also formulated and tested. A per-pixel weight map is computed from each patch's mask via
morphological dilation and erosion (their set difference isolates a thin ring at the true
boundary); boundary pixels are weighted 5x more heavily in the BCE term.

**Result:** trained for the same 10 epochs, this variant's best result (epoch 9) was IoU=0.610,
F1=0.754 -- modestly below the primary Dice+BCE run's best (IoU=0.642, F1=0.780). Since no
random seed was fixed for either run, part of this gap may reflect ordinary training
stochasticity (different random patch sampling and layer initialization) rather than a genuine
effect of boundary weighting, and the 5x weight/dilation radius were not tuned. The original,
unweighted Dice+BCE loss is therefore retained as the primary result reported above; this
variant is documented here as a tested, honestly-evaluated exploration rather than an
improvement.

In [ ]:
def compute_boundary_weight_map(mask, dilation_iterations=3, boundary_weight=5.0):
    """Returns a per-pixel weight map: elevated near building edges, 1.0 elsewhere."""
    mask_bool = mask.astype(bool)
    dilated = ndimage.binary_dilation(mask_bool, iterations=dilation_iterations)
    eroded = ndimage.binary_erosion(mask_bool, iterations=dilation_iterations)
    boundary = dilated & ~eroded
    weight_map = np.ones_like(mask, dtype=np.float32)
    weight_map[boundary] = boundary_weight
    return weight_map


class BoundaryWeightedDiceBCELoss(nn.Module):
    def __init__(self, bce_weight=0.5):
        super().__init__()
        self.bce_weight = bce_weight

    def forward(self, logits, targets, weight_map):
        bce_elementwise = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        weighted_bce = (bce_elementwise * weight_map).mean()

        probs = torch.sigmoid(logits)
        probs_flat = probs.view(-1)
        targets_flat = targets.view(-1)
        intersection = (probs_flat * targets_flat).sum()
        dice_coef = (2. * intersection + 1.0) / (probs_flat.sum() + targets_flat.sum() + 1.0)
        dice_loss = 1 - dice_coef

        return self.bce_weight * weighted_bce + (1 - self.bce_weight) * dice_loss

# NOTE: training this variant requires a modified tile_batch_generator that also yields the
# weight map per patch, and a training loop that unpacks/passes it into the loss -- see the
# accompanying report for the full comparison and results. Not re-run here since the primary
# Dice+BCE results above are the reported findings.